<a href="https://colab.research.google.com/github/Fordfire337/CS-4410-intro-machine-learning/blob/main/JWHW7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.datasets import mnist, fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dense, Flatten, MaxPooling2D
from tensorflow.keras.utils import to_categorical

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.19.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
SEED = 42
tf.keras.utils.set_random_seed(SEED)

EPOCHS = 5
BATCH_SIZE = 64
VALIDATION_SPLIT = 0.1
NUM_CLASSES = 10
IMAGE_SHAPE = (28, 28, 1)


In [3]:
def load_and_prepare(dataset_name):
    if dataset_name == "MNIST":
        (X_train, y_train), (X_test, y_test) = mnist.load_data()
    elif dataset_name == "Fashion-MNIST":
        (X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")

    X_train = X_train.reshape((X_train.shape[0], 28, 28, 1)).astype("float32") / 255.0
    X_test = X_test.reshape((X_test.shape[0], 28, 28, 1)).astype("float32") / 255.0

    y_train = to_categorical(y_train, NUM_CLASSES)
    y_test = to_categorical(y_test, NUM_CLASSES)

    return (X_train, y_train), (X_test, y_test)


In [4]:
def build_model(add_dense_4096=False, remove_first_dense=False):
    model = Sequential()

    model.add(Conv2D(filters=64, kernel_size=(3, 3), activation="relu", input_shape=IMAGE_SHAPE))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Conv2D(filters=128, kernel_size=(3, 3), activation="relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Flatten())

    if not remove_first_dense:
        model.add(Dense(units=128, activation="relu"))

    if add_dense_4096:
        model.add(Dense(units=4096, activation="relu"))

    model.add(Dense(units=10, activation="softmax"))

    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model


In [5]:
def train_and_evaluate(dataset_name, model_label, add_dense_4096=False, remove_first_dense=False):
    (X_train, y_train), (X_test, y_test) = load_and_prepare(dataset_name)
    model = build_model(add_dense_4096=add_dense_4096, remove_first_dense=remove_first_dense)

    start = time.perf_counter()
    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT,
        verbose=1
    )
    train_seconds = time.perf_counter() - start

    eval_start = time.perf_counter()
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)
    eval_seconds = time.perf_counter() - eval_start

    best_val_accuracy = max(history.history["val_accuracy"])
    final_val_accuracy = history.history["val_accuracy"][-1]
    final_train_accuracy = history.history["accuracy"][-1]

    total_params = model.count_params()

    return {
        "dataset": dataset_name,
        "model": model_label,
        "parameters": total_params,
        "train_accuracy_last_epoch": final_train_accuracy,
        "val_accuracy_last_epoch": final_val_accuracy,
        "best_val_accuracy": best_val_accuracy,
        "test_accuracy": test_accuracy,
        "test_loss": test_loss,
        "training_time_seconds": train_seconds,
        "evaluation_time_seconds": eval_seconds
    }


In [6]:
results = []

results.append(train_and_evaluate(
    dataset_name="MNIST",
    model_label="Baseline"
))

results.append(train_and_evaluate(
    dataset_name="Fashion-MNIST",
    model_label="Baseline"
))

results_df = pd.DataFrame(results)
results_df


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - accuracy: 0.9013 - loss: 0.3223 - val_accuracy: 0.9837 - val_loss: 0.0579
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9855 - loss: 0.0457 - val_accuracy: 0.9863 - val_loss: 0.0505
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9910 - loss: 0.0274 - val_accuracy: 0.9837 - val_loss: 0.0583
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9944 - loss: 0.0188 - val_accuracy: 0.9858 - val_loss: 0.0558
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9951 - loss: 0.0163 - val_accuracy: 0.9888 - val_loss: 0.0454
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9846 - loss: 0.0481
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.7641 - loss: 0.6462 - val_accuracy: 

,dataset,model,parameters,train_accuracy_last_epoch,val_accuracy_last_epoch,best_val_accuracy,test_accuracy,test_loss,training_time_seconds,evaluation_time_seconds
0,MNIST,Baseline,485514,0.995167,0.988833,0.988833,0.9873,0.039359,27.584809,1.752037
1,Fashion-MNIST,Baseline,485514,0.931000,0.913333,0.913333,0.9037,0.278398,24.937306,1.597245


In [7]:
results.append(train_and_evaluate(
    dataset_name="MNIST",
    model_label="Baseline + Dense(4096)",
    add_dense_4096=True
))

results.append(train_and_evaluate(
    dataset_name="Fashion-MNIST",
    model_label="Baseline + Dense(4096)",
    add_dense_4096=True
))

results_df = pd.DataFrame(results)
results_df


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.8967 - loss: 0.3234 - val_accuracy: 0.9798 - val_loss: 0.0673
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9847 - loss: 0.0477 - val_accuracy: 0.9853 - val_loss: 0.0521
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9905 - loss: 0.0304 - val_accuracy: 0.9858 - val_loss: 0.0534
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9924 - loss: 0.0239 - val_accuracy: 0.9882 - val_loss: 0.0488
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9945 - loss: 0.0182 - val_accuracy: 0.9893 - val_loss: 0.0381
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9868 - loss: 0.0471
Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.7457 - loss: 0.6694 - val_accuracy: 0.8605 - val_loss: 0.3810
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8832 - loss: 0.3153 - val_accuracy: 0.8795 - val_loss: 0.3397
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━

,dataset,model,parameters,train_accuracy_last_epoch,val_accuracy_last_epoch,best_val_accuracy,test_accuracy,test_loss,training_time_seconds,evaluation_time_seconds
0,MNIST,Baseline,485514,0.995167,0.988833,0.988833,0.9873,0.039359,27.584809,1.752037
1,Fashion-MNIST,Baseline,485514,0.931000,0.913333,0.913333,0.9037,0.278398,24.937306,1.597245
2,MNIST,Baseline + Dense(4096),1053578,0.994296,0.989333,0.989333,0.9890,0.037611,26.687023,1.663422
3,Fashion-MNIST,Baseline + Dense(4096),1053578,0.933870,0.896500,0.897333,0.8950,0.366657,25.909067,1.766421


In [8]:
summary_df = results_df.copy()

for col in ["train_accuracy_last_epoch", "val_accuracy_last_epoch", "best_val_accuracy", "test_accuracy"]:
    summary_df[col] = summary_df[col].map(lambda x: round(float(x), 4))

for col in ["test_loss", "training_time_seconds", "evaluation_time_seconds"]:
    summary_df[col] = summary_df[col].map(lambda x: round(float(x), 2))

summary_df = summary_df.sort_values(["dataset", "model"]).reset_index(drop=True)
summary_df


,dataset,model,parameters,train_accuracy_last_epoch,val_accuracy_last_epoch,best_val_accuracy,test_accuracy,test_loss,training_time_seconds,evaluation_time_seconds
0,Fashion-MNIST,Baseline,485514,0.9310,0.9133,0.9133,0.9037,0.28,24.94,1.60
1,Fashion-MNIST,Baseline + Dense(4096),1053578,0.9339,0.8965,0.8973,0.8950,0.37,25.91,1.77
2,MNIST,Baseline,485514,0.9952,0.9888,0.9888,0.9873,0.04,27.58,1.75
3,MNIST,Baseline + Dense(4096),1053578,0.9943,0.9893,0.9893,0.9890,0.04,26.69,1.66
